In [312]:
import os

folder = "data/raw"

files = os.listdir(folder)

print(f"Found {len(files)} files:\n")
for f in files:
    print(f)

Found 20 files:

ALQ_L.xlsx
BIOPRO_L.xlsx
BMX_L.xlsx
BPQ_L.xlsx
BPXO_L.xlsx
DEMO_L.xlsx
DIQ_L.xlsx
DPQ_L.xlsx
FASTQX_L.xlsx
GHB_L.xlsx
GLU_L.xlsx
HDL_L.xlsx
HSCRP_L.xlsx
INS_L.xlsx
MCQ_L.xlsx
PAQ_L.xlsx
SLQ_L.xlsx
SMQ_L.xlsx
TCHOL_L.xlsx
TRIGLY_L.xlsx


In [313]:
import pandas as pd

folder = "data/raw"

# Load each file into a dictionary
data = {}

for file in files:
    name = file.replace(".xlsx", "")        # e.g. "ALQ_L.xlsx" -> "ALQ_L"
    path = os.path.join(folder, file)
    data[name] = pd.read_excel(path)
    print(f"Loaded {name:12} -> {data[name].shape[0]} rows, {data[name].shape[1]} columns")

print("\nAll files loaded into 'data' dictionary!")

Loaded ALQ_L        -> 6337 rows, 9 columns
Loaded BIOPRO_L     -> 7199 rows, 42 columns
Loaded BMX_L        -> 8860 rows, 22 columns
Loaded BPQ_L        -> 8501 rows, 6 columns
Loaded BPXO_L       -> 7801 rows, 12 columns
Loaded DEMO_L       -> 11933 rows, 27 columns
Loaded DIQ_L        -> 11744 rows, 9 columns
Loaded DPQ_L        -> 6337 rows, 11 columns
Loaded FASTQX_L     -> 8727 rows, 19 columns
Loaded GHB_L        -> 7199 rows, 3 columns
Loaded GLU_L        -> 3996 rows, 4 columns
Loaded HDL_L        -> 8068 rows, 4 columns
Loaded HSCRP_L      -> 8727 rows, 4 columns
Loaded INS_L        -> 3996 rows, 5 columns
Loaded MCQ_L        -> 11744 rows, 35 columns
Loaded PAQ_L        -> 8153 rows, 8 columns
Loaded SLQ_L        -> 8501 rows, 7 columns
Loaded SMQ_L        -> 9015 rows, 9 columns
Loaded TCHOL_L      -> 8068 rows, 4 columns
Loaded TRIGLY_L     -> 3996 rows, 10 columns

All files loaded into 'data' dictionary!


In [314]:
# Look at the demographics table
demo = data["DEMO_L"]

print("Shape:", demo.shape)   # rows, columns
print("\nFirst 5 rows:")
demo.head()

Shape: (11933, 27)

First 5 rows:


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378,12,2,1,43,0,5,6,2,0,...,0,0,0,0,0,50055.450807,54374.463898,173,2,5.00
1,130379,12,2,1,66,0,3,3,2,0,...,0,0,0,0,0,29087.450605,34084.721548,173,2,5.00
2,130380,12,2,2,44,0,2,2,1,0,...,0,0,0,0,0,80062.674301,81196.277992,174,1,1.41
3,130381,12,2,2,5,0,5,7,1,71,...,2,2,2,3,0,38807.268902,55698.607106,182,2,1.53
4,130382,12,2,1,2,0,3,3,2,34,...,2,2,3,1,2,30607.519774,36434.146346,182,2,3.60


In [315]:
# Keep only the demographics columns we need
demo_clean = data["DEMO_L"][[
    "SEQN",       # unique ID
    "RIDAGEYR",   # age
    "RIAGENDR",   # gender
    "RIDRETH3",   # race
    "INDFMPIR",   # income-to-poverty ratio
    "DMDEDUC2",   # education
    "RIDEXPRG"    # pregnancy status (for filtering only - NOT a feature)
]]

print("Shape:", demo_clean.shape)
demo_clean.head()

Shape: (11933, 7)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,INDFMPIR,DMDEDUC2,RIDEXPRG
0,130378,43,1,6,5.00,5,0
1,130379,66,1,3,5.00,5,0
2,130380,44,2,2,1.41,3,2
3,130381,5,2,7,1.53,0,0
4,130382,2,1,3,3.60,0,0


## DEMO_L — Demographics Table

This table has **27 columns**, but we only keep **6** useful ones.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (used to join all tables) |
| RIDAGEYR | Age in years |
| RIAGENDR | Gender |
| RIDRETH3 | Race / ethnicity |
| INDFMPIR | Income-to-poverty ratio |
| DMDEDUC2 | Education level |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| SDDSRVYR | Survey cycle number | Just a label, not health |
| RIDSTATR | Exam completion status | Used to filter rows only, not a feature |
| RIDAGEMN | Age in months | Duplicate — we already have age in years |
| RIDEXAGM | Exam age in months | Duplicate of age |
| RIDRETH1 | Old race grouping | We use the newer RIDRETH3 instead |
| RIDEXMON | Exam season | Not a health risk factor |
| DMDBORN4 | Country of birth | Weak predictor |
| DMDYRUSZ | Years living in US | Weak predictor |
| DMDMARTZ | Marital status | Weak predictor, optional |
| RIDEXPRG | Pregnancy status | Used to filter out pregnant women only |
| DMDHRGND | Household head gender | About another person, not the participant |
| DMDHRAGZ | Household head age | About another person |
| DMDHREDZ | Household head education | About another person |
| DMDHRMAZ | Household head marital status | About another person |
| DMDHSEDZ | Spouse education | About another person |
| WTINT2YR | Survey interview weight | Statistical sampling weight, not health |
| WTMEC2YR | Survey exam weight | Statistical sampling weight, not health |
| SDMVSTRA | Survey strata code | Statistics admin code |
| SDMVPSU | Survey cluster code | Statistics admin code |

**Rule:** Keep real health-risk info. Ignore survey codes, weights, duplicate columns, and info about other household members.


In [316]:
# Look at the body measurements table
bmx = data["BMX_L"]

print("Shape:", bmx.shape)
print("\nFirst 5 rows:")
bmx.head()

Shape: (8860, 22)

First 5 rows:


,SEQN,BMDSTATS,BMXWT,BMIWT,BMXRECUM,BMIRECUM,BMXHEAD,BMIHEAD,BMXHT,BMIHT,...,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,130378,1,86.9,0,0.0,0,0.0,0,179.5,0,...,42.8,0,42.0,0,35.7,0,98.3,0,102.9,0
1,130379,1,101.8,0,0.0,0,0.0,0,174.2,0,...,38.5,0,38.7,0,33.7,0,114.7,0,112.4,0
2,130380,1,69.4,0,0.0,0,0.0,0,152.9,0,...,38.5,0,35.5,0,36.3,0,93.5,0,98.0,0
3,130381,1,34.3,0,0.0,0,0.0,0,120.1,0,...,0.0,0,25.4,0,23.4,0,70.4,0,0.0,0
4,130382,3,13.6,0,0.0,1,0.0,0,0.0,1,...,0.0,0,0.0,1,0.0,1,0.0,1,0.0,0


In [317]:
# Keep only the body measurement columns we need
bmx_clean = data["BMX_L"][[
    "SEQN",       # unique ID
    "BMXWT",      # weight (kg)
    "BMXHT",      # height (cm)
    "BMXWAIST",   # waist (cm)
    "BMXBMI"      # body mass index
]]

print("Shape:", bmx_clean.shape)
bmx_clean.head()

Shape: (8860, 5)


,SEQN,BMXWT,BMXHT,BMXWAIST,BMXBMI
0,130378,86.9,179.5,98.3,27.0
1,130379,101.8,174.2,114.7,33.5
2,130380,69.4,152.9,93.5,29.7
3,130381,34.3,120.1,70.4,23.8
4,130382,13.6,0.0,0.0,0.0


## BMX_L — Body Measurements Table

This table has **22 columns**, but we only keep **4** useful ones.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| BMXWT | Body weight (kg) |
| BMXHT | Height (cm) |
| BMXWAIST | Waist circumference (cm) |
| BMXBMI | Body Mass Index (used to build obesity label) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| BMDSTATS | Measurement status flag | Quality flag, not a feature |
| BMIWT | Weight comment flag | Just a flag (0/1), not a value |
| BMXRECUM | Recumbent length (babies) | For infants — adults are blank |
| BMIRECUM | Recumbent flag | Flag only |
| BMXHEAD | Head circumference (babies) | For infants only |
| BMIHEAD | Head flag | Flag only |
| BMIHT | Height comment flag | Flag only |
| BMDBMIC | BMI category (children) | For children only |
| BMXLEG | Upper leg length | Not needed for our targets |
| BMILEG | Leg flag | Flag only |
| BMXARML | Arm length | Not needed |
| BMIARML | Arm length flag | Flag only |
| BMXARMC | Arm circumference | Not needed |
| BMIARMC | Arm circ flag | Flag only |
| BMIWAIST | Waist comment flag | Flag only |
| BMXHIP | Hip circumference | We use waist instead |
| BMIHIP | Hip flag | Flag only |

**Rule:** Keep weight, height, waist, BMI. Ignore baby measurements, arm/leg sizes, and all the comment flag columns.

In [318]:
# Look at the blood pressure table
bpxo = data["BPXO_L"]

print("Shape:", bpxo.shape)
print("\nFirst 5 rows:")
bpxo.head()

Shape: (7801, 12)

First 5 rows:


,SEQN,BPAOARM,BPAOCSZ,BPXOSY1,BPXODI1,BPXOSY2,BPXODI2,BPXOSY3,BPXODI3,BPXOPLS1,BPXOPLS2,BPXOPLS3
0,130378,R,4,135,98,131,96,132,94,82,79,82
1,130379,R,4,121,84,117,76,113,76,72,71,73
2,130380,R,4,111,79,112,80,104,76,84,83,77
3,130386,R,4,110,72,120,74,115,75,59,64,64
4,130387,R,4,143,76,136,74,145,78,80,80,77


In [319]:
# Keep blood pressure readings (all 3 times)
bpxo_clean = data["BPXO_L"][[
    "SEQN",       # unique ID
    "BPXOSY1", "BPXOSY2", "BPXOSY3",   # systolic (top number) x3
    "BPXODI1", "BPXODI2", "BPXODI3",   # diastolic (bottom number) x3
    "BPXOPLS1", "BPXOPLS2", "BPXOPLS3" # pulse x3
]]

print("Shape:", bpxo_clean.shape)
bpxo_clean.head()

Shape: (7801, 10)


,SEQN,BPXOSY1,BPXOSY2,BPXOSY3,BPXODI1,BPXODI2,BPXODI3,BPXOPLS1,BPXOPLS2,BPXOPLS3
0,130378,135,131,132,98,96,94,82,79,82
1,130379,121,117,113,84,76,76,72,71,73
2,130380,111,112,104,79,80,76,84,83,77
3,130386,110,120,115,72,74,75,59,64,64
4,130387,143,136,145,76,74,78,80,80,77


## BPXO_L — Blood Pressure Table

This table has **12 columns**. Blood pressure was measured **3 times** for accuracy — we keep all 3 readings and will average them later.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| BPXOSY1, BPXOSY2, BPXOSY3 | Systolic BP (top number), 3 readings |
| BPXODI1, BPXODI2, BPXODI3 | Diastolic BP (bottom number), 3 readings |
| BPXOPLS1, BPXOPLS2, BPXOPLS3 | Pulse rate, 3 readings |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| BPAOARM | Which arm used (L/R) | Procedure detail, not a health risk |
| BPAOCSZ | Cuff size used | Procedure detail, not a health risk |

**Rule:** Keep all 3 blood pressure and pulse readings (we average them later). Ignore the procedure details like arm and cuff size.

In [320]:
# Look at the HbA1c table
ghb = data["GHB_L"]

print("Shape:", ghb.shape)
print("\nFirst 5 rows:")
ghb.head()

Shape: (7199, 3)

First 5 rows:


,SEQN,WTPH2YR,LBXGH
0,130378,56042.129410,5.6
1,130379,37435.705647,5.6
2,130380,85328.844519,6.2
3,130386,44526.214135,5.1
4,130387,22746.296353,5.9


In [321]:
# Keep only HbA1c (used to build the diabetes label)
ghb_clean = data["GHB_L"][[
    "SEQN",     # unique ID
    "LBXGH"     # HbA1c % (diabetes label maker)
]]

print("Shape:", ghb_clean.shape)
ghb_clean.head()

Shape: (7199, 2)


,SEQN,LBXGH
0,130378,5.6
1,130379,5.6
2,130380,6.2
3,130386,5.1
4,130387,5.9


## GHB_L — HbA1c (Glycohemoglobin) Table

This table has **3 columns**. HbA1c = average blood sugar over 3 months — the **main diabetes test**.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXGH | HbA1c % (used to BUILD the diabetes label) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTPH2YR | Survey sampling weight | Statistical weight, not health data |

### ⚠️ Important — LEAKAGE WARNING

`LBXGH` (HbA1c) is a **LABEL maker**, not a feature.

- We use it to CREATE the diabetes answer (Yes/No)
- Then we **DROP it** from the features
- **Reason:** HbA1c IS the diabetes test. Using it to predict diabetes = cheating = fake 100% accuracy

**Rule:** Keep HbA1c to build the label, then remove it before training the diabetes model.

In [322]:
# Look at the glucose table
glu = data["GLU_L"]

print("Shape:", glu.shape)
print("\nFirst 5 rows:")
glu.head()

Shape: (3996, 4)

First 5 rows:


,SEQN,WTSAF2YR,LBXGLU,LBDGLUSI
0,130378,120025.308444,113,6.27
1,130379,0.000000,99,5.50
2,130380,145090.773569,156,8.66
3,130386,82599.618089,100,5.55
4,130394,100420.348913,88,4.88


In [323]:
# Keep only glucose (used to build diabetes + metabolic labels)
glu_clean = data["GLU_L"][[
    "SEQN",      # unique ID
    "LBXGLU"     # fasting glucose mg/dL (label maker)
]]

print("Shape:", glu_clean.shape)
glu_clean.head()

Shape: (3996, 2)


,SEQN,LBXGLU
0,130378,113
1,130379,99
2,130380,156
3,130386,100
4,130394,88


## GLU_L — Fasting Glucose Table

This table has **4 columns**. Glucose = blood sugar after fasting. Only **3,996 people** have this (only those who fasted 8+ hours).

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXGLU | Fasting glucose mg/dL (used to BUILD diabetes + metabolic labels) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTSAF2YR | Fasting survey sampling weight | Statistical weight, not health |
| LBDGLUSI | Glucose in mmol/L | Duplicate — same value, different unit |

### ⚠️ Important — LEAKAGE WARNING

`LBXGLU` (glucose) is a **LABEL maker**, not a feature.

- Used to BUILD the diabetes label AND the metabolic syndrome label
- Then we **DROP it** before training those models
- **Reason:** Glucose IS used to diagnose diabetes — using it to predict diabetes = cheating

**Rule:** Keep glucose to build the labels, then remove it before training diabetes and metabolic syndrome models.

In [324]:
# Look at the insulin table
ins = data["INS_L"]

print("Shape:", ins.shape)
print("\nFirst 5 rows:")
ins.head()

Shape: (3996, 5)

First 5 rows:


,SEQN,WTSAF2YR,LBXIN,LBDINSI,LBDINLC
0,130378,120025.308444,15.53,93.18,0
1,130379,0.000000,19.91,119.46,0
2,130380,145090.773569,16.33,97.98,0
3,130386,82599.618089,11.38,68.28,0
4,130394,100420.348913,7.20,43.20,0


In [325]:
# Keep only insulin (note: this is LEAK - we drop it before training)
ins_clean = data["INS_L"][[
    "SEQN",     # unique ID
    "LBXIN"     # insulin (LEAK - drop before training)
]]

print("Shape:", ins_clean.shape)
ins_clean.head()

Shape: (3996, 2)


,SEQN,LBXIN
0,130378,15.53
1,130379,19.91
2,130380,16.33
3,130386,11.38
4,130394,7.20


## INS_L — Insulin Table

This table has **5 columns**. Insulin = the hormone that controls blood sugar. Only **3,996 people** (fasting required).

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXIN | Insulin µU/mL |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTSAF2YR | Fasting survey sampling weight | Statistical weight, not health |
| LBDINSI | Insulin in pmol/L | Duplicate — same value, different unit |
| LBDINLC | Comment / quality flag | Flag only, not a value |

### ⚠️ Important — LEAKAGE WARNING

`LBXIN` (insulin) is **LEAK** — we never use it as a feature.

- Insulin is too directly tied to blood sugar control
- We **DROP it everywhere** (especially the diabetes model)
- **Reason:** It almost directly reveals diabetes status = cheating

**Rule:** We keep it loaded for now, but it gets dropped before training. Best to treat it as "never a feature."

### Optional note (HOMA-IR)
In our OLD wrong project we combined glucose × insulin to make "HOMA-IR". We are NOT doing that now — it caused leakage. We keep things clean and honest.

In [326]:
# Look at the HDL cholesterol table
hdl = data["HDL_L"]

print("Shape:", hdl.shape)
print("\nFirst 5 rows:")
hdl.head()

Shape: (8068, 4)

First 5 rows:


,SEQN,WTPH2YR,LBDHDD,LBDHDDSI
0,130378,56042.129410,45,1.16
1,130379,37435.705647,60,1.55
2,130380,85328.844519,49,1.27
3,130386,44526.214135,46,1.19
4,130387,22746.296353,42,1.09


In [327]:
# Keep only HDL cholesterol (safe feature)
hdl_clean = data["HDL_L"][[
    "SEQN",      # unique ID
    "LBDHDD"     # HDL cholesterol mg/dL (good cholesterol)
]]

print("Shape:", hdl_clean.shape)
hdl_clean.head()

Shape: (8068, 2)


,SEQN,LBDHDD
0,130378,45
1,130379,60
2,130380,49
3,130386,46
4,130387,42


## HDL_L — HDL Cholesterol Table (Good Cholesterol)

This table has **4 columns**. HDL = the "good" cholesterol that protects the heart.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBDHDD | HDL cholesterol mg/dL (good cholesterol) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTPH2YR | Survey sampling weight | Statistical weight, not health |
| LBDHDDSI | HDL in mmol/L | Duplicate — same value, different unit |

### ✅ Safe feature

`LBDHDD` (HDL) is a **safe feature** — it does NOT directly reveal any disease.

- Use it freely for diabetes, CVD, hypertension models
- **One exception:** for the **metabolic syndrome** model, HDL is one of the 5 criteria → drop it there only

**Rule:** Keep HDL as a feature everywhere, except drop it when predicting metabolic syndrome.

In [328]:
# Look at the total cholesterol table
tchol = data["TCHOL_L"]

print("Shape:", tchol.shape)
print("\nFirst 5 rows:")
tchol.head()

Shape: (8068, 4)

First 5 rows:


,SEQN,WTPH2YR,LBXTC,LBDTCSI
0,130378,56042.129410,264,6.83
1,130379,37435.705647,214,5.53
2,130380,85328.844519,187,4.84
3,130386,44526.214135,183,4.73
4,130387,22746.296353,203,5.25


In [329]:
# Keep only total cholesterol (safe feature for ALL models)
tchol_clean = data["TCHOL_L"][[
    "SEQN",      # unique ID
    "LBXTC"      # total cholesterol mg/dL
]]

print("Shape:", tchol_clean.shape)
tchol_clean.head()

Shape: (8068, 2)


,SEQN,LBXTC
0,130378,264
1,130379,214
2,130380,187
3,130386,183
4,130387,203


## TCHOL_L — Total Cholesterol Table

This table has **4 columns**. Total cholesterol = the overall cholesterol level in blood.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXTC | Total cholesterol mg/dL |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTPH2YR | Survey sampling weight | Statistical weight, not health |
| LBDTCSI | Total cholesterol in mmol/L | Duplicate — same value, different unit |

### ✅ Safest feature

`LBXTC` (total cholesterol) is the **safest feature** — use it for ALL 5 models.

- It is NOT used to define any of our targets
- No leakage risk anywhere

**Rule:** Keep total cholesterol as a feature for every model. No exceptions.

In [330]:
# Look at the triglycerides + LDL table
trigly = data["TRIGLY_L"]

print("Shape:", trigly.shape)
print("\nFirst 5 rows:")
trigly.head()

Shape: (3996, 10)

First 5 rows:


,SEQN,WTSAF2YR,LBXTLG,LBDTRSI,LBDLDL,LBDLDLSI,LBDLDLM,LBDLDMSI,LBDLDLN,LBDLDNSI
0,130378,120025.308444,153,1.727,188,4.862,190,4.913,191,4.939
1,130379,0.000000,86,0.971,137,3.543,135,3.491,139,3.595
2,130380,145090.773569,375,4.234,63,1.629,90,2.327,78,2.017
3,130386,82599.618089,142,1.603,109,2.819,111,2.870,112,2.896
4,130394,100420.348913,57,0.644,124,3.207,120,3.103,124,3.207


In [331]:
# Keep triglycerides and main LDL (bad cholesterol)
trigly_clean = data["TRIGLY_L"][[
    "SEQN",       # unique ID
    "LBXTLG",     # triglycerides mg/dL
    "LBDLDL"      # LDL cholesterol mg/dL (bad cholesterol)
]]

print("Shape:", trigly_clean.shape)
trigly_clean.head()

Shape: (3996, 3)


,SEQN,LBXTLG,LBDLDL
0,130378,153,188
1,130379,86,137
2,130380,375,63
3,130386,142,109
4,130394,57,124


## TRIGLY_L — Triglycerides + LDL Table

This table has **10 columns**. It contains triglycerides (blood fat) and LDL (bad cholesterol). Many columns are duplicate units or alternate LDL formulas — we keep just **2**.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXTLG | Triglycerides mg/dL (blood fat) |
| LBDLDL | LDL cholesterol mg/dL (bad cholesterol) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTSAF2YR | Fasting survey weight | Statistical weight, not health |
| LBDTRSI | Triglycerides in mmol/L | Duplicate — same value, different unit |
| LBDLDLSI | LDL in mmol/L | Duplicate unit |
| LBDLDLM | LDL (alternate formula) | Extra LDL method — keep just LBDLDL |
| LBDLDMSI | LDL alt formula in mmol/L | Duplicate unit |
| LBDLDLN | LDL (another formula) | Extra LDL method |
| LBDLDNSI | LDL another formula in mmol/L | Duplicate unit |

### ✅ Mostly safe features

- `LBDLDL` (LDL) is **safe** for all models
- `LBXTLG` (triglycerides) is **safe** for most models — but it is one of the 5 metabolic syndrome criteria → drop it only for the metabolic syndrome model

**Rule:** Keep triglycerides and LDL as features. Drop triglycerides only when predicting metabolic syndrome.

In [332]:
# Look at the C-Reactive Protein (inflammation) table
hscrp = data["HSCRP_L"]

print("Shape:", hscrp.shape)
print("\nFirst 5 rows:")
hscrp.head()

Shape: (8727, 4)

First 5 rows:


,SEQN,WTPH2YR,LBXHSCRP,LBDHRPLC
0,130378,56042.129410,1.78,0
1,130379,37435.705647,2.03,0
2,130380,85328.844519,5.62,0
3,130381,0.000000,0.00,0
4,130382,59638.932323,0.00,0


In [333]:
# Keep only CRP (safe inflammation feature for ALL models)
hscrp_clean = data["HSCRP_L"][[
    "SEQN",         # unique ID
    "LBXHSCRP"      # C-Reactive Protein mg/L (inflammation)
]]

print("Shape:", hscrp_clean.shape)
hscrp_clean.head()

Shape: (8727, 2)


,SEQN,LBXHSCRP
0,130378,1.78
1,130379,2.03
2,130380,5.62
3,130381,0.00
4,130382,0.00


## HSCRP_L — C-Reactive Protein Table (Inflammation)

This table has **4 columns**. CRP = a protein that rises when there is inflammation in the body. High CRP links to heart disease and diabetes risk.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXHSCRP | C-Reactive Protein mg/L (inflammation marker) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTPH2YR | Survey sampling weight | Statistical weight, not health |
| LBDHRPLC | Comment / quality flag | Flag only, not a value |

### ✅ Safe feature

`LBXHSCRP` (CRP) is a **safe feature** — use it for ALL 5 models.

- It is NOT used to define any of our targets
- It is a general risk marker, no leakage

**Rule:** Keep CRP as a feature for every model. No exceptions.

In [334]:
# Look at the biochemistry panel (kidney/liver) table
biopro = data["BIOPRO_L"]

print("Shape:", biopro.shape)
print("\nColumn names:")
print(list(biopro.columns))

Shape: (7199, 42)

Column names:
['SEQN', 'WTPH2YR', 'LBXSATSI', 'LBXSAL', 'LBDSALSI', 'LBXSAPSI', 'LBXSASSI', 'LBXSC3SI', 'LBXSBU', 'LBDSBUSI', 'LBXSCLSI', 'LBXSCK', 'LBXSCR', 'LBDSCRSI', 'LBXSGB', 'LBDSGBSI', 'LBXSGL', 'LBDSGLSI', 'LBXSGTSI', 'LBDSGTLC', 'LBXSIR', 'LBDSIRSI', 'LBXSLDSI', 'LBXMAGN', 'LBXSOSSI', 'LBXSPH', 'LBDSPHSI', 'LBXSKSI', 'LBXSNASI', 'LBXSTB', 'LBDSTBSI', 'LBDSTBLC', 'LBXSCA', 'LBDSCASI', 'LBXSCH', 'LBDSCHSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI']


In [335]:
# Keep only a few safe kidney/liver markers
# NOTE: LBXSGL (glucose) is in this table but we DO NOT keep it - it's LEAK
biopro_clean = data["BIOPRO_L"][[
    "SEQN",       # unique ID
    "LBXSCR",     # creatinine (kidney function)
    "LBXSUA",     # uric acid
    "LBXSBU",     # blood urea nitrogen (kidney)
    "LBXSAL"      # albumin (liver/nutrition)
]]

print("Shape:", biopro_clean.shape)
biopro_clean.head()

Shape: (7199, 5)


,SEQN,LBXSCR,LBXSUA,LBXSBU,LBXSAL
0,130378,0.80,5.1,11,4.3
1,130379,0.79,8.5,24,3.9
2,130380,0.64,4.4,10,3.7
3,130386,0.82,6.0,17,4.3
4,130387,0.76,6.2,15,3.7


## BIOPRO_L — Biochemistry Panel Table (Kidney / Liver)

This table has **42 columns** — the biggest table! Most are duplicate SI units. We keep only **4 safe, useful markers**.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| LBXSCR | Creatinine (kidney function marker) |
| LBXSUA | Uric acid (linked to metabolic risk) |
| LBXSBU | Blood urea nitrogen (kidney marker) |
| LBXSAL | Albumin (liver / nutrition marker) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| WTPH2YR | Survey sampling weight | Statistical weight, not health |
| LBXSGL | Glucose (in this panel!) | ⚠️ LEAK — same as fasting glucose, drop for diabetes |
| LBXSCH | Cholesterol (in panel) | Duplicate of TCHOL — avoid double counting |
| LBXSTR | Triglycerides (in panel) | Duplicate of TRIGLY |
| All `...SI` columns | Same values in SI units | Duplicates — same number, different unit |
| LBXSATSI, LBXSASSI, etc. | Liver enzymes, minerals | Optional extras — skip to keep it simple |

### ✅ Safe features (the 4 we kept)

Creatinine, uric acid, BUN, and albumin are **safe risk factors** — they do not define any of our 5 targets.

### ⚠️ One leakage trap
`LBXSGL` (glucose) hides in this table. We do NOT keep it — it's the same as the glucose in GLU_L and would leak the diabetes answer.

**Rule:** Keep the 4 kidney/liver markers. Skip all duplicate units, the hidden glucose, and the optional enzymes.

In [336]:
# Look at the fasting questionnaire table
fastqx = data["FASTQX_L"]

print("Shape:", fastqx.shape)
print("\nFirst 5 rows:")
fastqx.head()

Shape: (8727, 19)

First 5 rows:


,SEQN,PHQ020,PHACOFHR,PHACOFMN,PHQ030,PHAALCHR,PHAALCMN,PHQ040,PHAGUMHR,PHAGUMMN,PHQ050,PHAANTHR,PHAANTMN,PHQ060,PHASUPHR,PHASUPMN,PHAFSTHR,PHAFSTMN,PHDSESNZ
0,130378,2,0,0,2,0,0,2,0,0,2,0,0,2,0,0,14,3,0
1,130379,2,0,0,2,0,0,2,0,0,2,0,0,2,0,0,2,45,0
2,130380,2,0,0,2,0,0,2,0,0,2,0,0,2,0,0,11,11,0
3,130381,2,0,0,2,0,0,2,0,0,2,0,0,2,0,0,4,9,1
4,130382,2,0,0,2,0,0,2,0,0,2,0,0,2,0,0,0,1,0


In [337]:
# Keep fasting hours (used to validate glucose/insulin readings)
fastqx_clean = data["FASTQX_L"][[
    "SEQN",        # unique ID
    "PHAFSTHR"     # fasting hours (how long since last ate)
]]

print("Shape:", fastqx_clean.shape)
fastqx_clean.head()

Shape: (8727, 2)


,SEQN,PHAFSTHR
0,130378,14
1,130379,2
2,130380,11
3,130381,4
4,130382,0


## FASTQX_L — Fasting Questionnaire Table (Lab Support)

This table has **19 columns**. It tells us how long each person fasted before their blood test. Glucose and insulin are only valid if the person fasted 8+ hours.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| PHAFSTHR | Fasting hours (how long since last ate) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| PHAFSTMN | Fasting minutes | We use hours; minutes optional |
| PHQ020 / PHACOFHR / PHACOFMN | Coffee questions | Sub-questions, not needed |
| PHQ030 / PHAALCHR / PHAALCMN | Alcohol before test | Not needed |
| PHQ040 / PHAGUMHR / PHAGUMMN | Gum chewing | Not needed |
| PHQ050 / PHAANTHR / PHAANTMN | Antacid taken | Not needed |
| PHQ060 / PHASUPHR / PHASUPMN | Supplements taken | Not needed |
| PHDSESNZ | Session flag | Admin flag |

### ✅ Special — never causes leakage

`PHAFSTHR` (fasting hours) is **NOT a disease value** — it's just "how long since you ate." So it never causes leakage.

### How we'll use it later
We use it only to CHECK/CLEAN glucose and insulin:
- If fasting hours ≥ 8 → glucose reading is valid ✅
- If fasting hours < 8 → glucose may be unreliable

It's a cleaning helper, not really a feature.

**Rule:** Keep fasting hours to validate glucose/insulin. Ignore all the coffee/gum/alcohol sub-questions.

In [338]:
# Look at the diabetes diagnosis table
diq = data["DIQ_L"]

print("Shape:", diq.shape)
print("\nFirst 5 rows:")
diq.head()

Shape: (11744, 9)

First 5 rows:


,SEQN,DIQ010,DID040,DIQ160,DIQ180,DIQ050,DID060,DIQ060U,DIQ070
0,130378,2,0,2,2,0,0,0,0
1,130379,2,0,2,1,0,0,0,0
2,130380,1,35,0,0,2,0,0,1
3,130381,2,0,0,0,0,0,0,0
4,130382,2,0,0,0,0,0,0,0


In [339]:
# Keep diabetes diagnosis columns (used to BUILD the diabetes label)
diq_clean = data["DIQ_L"][[
    "SEQN",      # unique ID
    "DIQ010",    # told you have diabetes? (1=Yes, 2=No, 3=Borderline)
    "DIQ160"     # told you have prediabetes? (helps prediabetes label)
]]

print("Shape:", diq_clean.shape)
diq_clean.head()

Shape: (11744, 3)


,SEQN,DIQ010,DIQ160
0,130378,2,2
1,130379,2,2
2,130380,1,0
3,130381,2,0
4,130382,2,0


## DIQ_L — Diabetes Diagnosis Table (LABEL MAKER)

This table has **9 columns**. It holds the doctor's diabetes diagnosis. We use it ONLY to create the diabetes answer (label).

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| DIQ010 | Told you have diabetes? (1=Yes, 2=No, 3=Borderline, 9=Don't know) |
| DIQ160 | Told you have prediabetes? (helps build prediabetes label) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| DID040 | Age when diagnosed | ⚠️ LEAK — only diabetics have this |
| DIQ050 | Taking insulin now? | ⚠️ LEAK — treatment reveals disease |
| DID060 | How long taking insulin | LEAK — treatment detail |
| DIQ060U | Insulin time unit | Treatment detail |
| DIQ070 | Taking diabetes pills? | ⚠️ LEAK — treatment reveals disease |
| DIQ180 | Had blood test recently? | Not needed |

### ⚠️ Important — this is a LABEL MAKER

- `DIQ010` is used to CREATE the diabetes answer (Yes/No)
- After building the label, we **DROP it** from features
- We also drop the treatment columns (insulin, pills) — they reveal the disease

### Special codes to clean later
- DIQ010 = **9** means "Don't know" → remove these rows (only a few)

**Rule:** Use DIQ010 to build the diabetes label, then drop it. Never use diagnosis or treatment columns as features.

In [340]:
# Look at the blood pressure diagnosis table
bpq = data["BPQ_L"]

print("Shape:", bpq.shape)
print("\nFirst 5 rows:")
bpq.head()

Shape: (8501, 6)

First 5 rows:


,SEQN,BPQ020,BPQ030,BPQ150,BPQ080,BPQ101D
0,130378,1,1,1,2,2
1,130379,1,1,1,2,2
2,130380,2,0,0,1,1
3,130384,2,0,0,2,2
4,130385,2,0,0,2,2


In [341]:
# Keep blood pressure diagnosis (used to BUILD the hypertension label)
bpq_clean = data["BPQ_L"][[
    "SEQN",      # unique ID
    "BPQ020"     # told you have high BP? (1=Yes, 2=No)
]]

print("Shape:", bpq_clean.shape)
bpq_clean.head()

Shape: (8501, 2)


,SEQN,BPQ020
0,130378,1
1,130379,1
2,130380,2
3,130384,2
4,130385,2


## BPQ_L — Blood Pressure Diagnosis Table (LABEL MAKER)

This table has **6 columns**. It holds the doctor's high blood pressure diagnosis. We use it ONLY to create the hypertension answer (label).

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| BPQ020 | Told you have high blood pressure? (1=Yes, 2=No) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| BPQ030 | Told high BP two or more times | Diagnosis detail, not needed |
| BPQ150 | Told to take aspirin | Treatment detail |
| BPQ080 | Taking BP-lowering medication? | ⚠️ LEAK — treatment reveals disease |
| BPQ101D | Now taking prescribed BP meds | ⚠️ LEAK — treatment reveals disease |

### ⚠️ Important — this is a LABEL MAKER

- `BPQ020` is used to CREATE the hypertension answer (Yes/No)
- After building the label, we **DROP it** from features
- BP medication columns are LEAK — being on BP meds means they have hypertension

### Special codes to clean later
- BPQ020 = **7** (Refused) or **9** (Don't know) → remove these rows

**Rule:** Use BPQ020 to build the hypertension label, then drop it. Never use BP medication as a feature.

In [342]:
# Look at the medical conditions table
mcq = data["MCQ_L"]

print("Shape:", mcq.shape)
print("\nColumn names:")
print(list(mcq.columns))

Shape: (11744, 35)

Column names:
['SEQN', 'MCQ010', 'MCQ035', 'MCQ040', 'MCQ050', 'AGQ030', 'MCQ053', 'MCQ149', 'MCQ160A', 'MCQ195', 'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ160M', 'MCQ170M', 'MCQ160P', 'MCQ160L', 'MCQ170L', 'MCQ500', 'MCQ510A', 'MCQ510B', 'MCQ510C', 'MCQ510D', 'MCQ510E', 'MCQ510F', 'MCQ550', 'MCQ560', 'MCQ220', 'MCQ230A', 'MCQ230B', 'MCQ230C', 'MCQ230D', 'OSQ230']


In [343]:
# Keep heart disease columns (used to BUILD the CVD label)
mcq_clean = data["MCQ_L"][[
    "SEQN",       # unique ID
    "MCQ160E",    # ever had heart attack? (1=Yes, 2=No)
    "MCQ160F"     # ever had coronary heart disease? (1=Yes, 2=No)
]]

print("Shape:", mcq_clean.shape)
mcq_clean.head()

Shape: (11744, 3)


,SEQN,MCQ160E,MCQ160F
0,130378,2,2
1,130379,2,2
2,130380,2,2
3,130381,0,0
4,130382,0,0


## MCQ_L — Medical Conditions Table (LABEL MAKER)

This table has **35 columns** about many different diseases. We only need **2** — the heart questions — to build the CVD (heart) label.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| MCQ160E | Ever told you had a heart attack? (1=Yes, 2=No) |
| MCQ160F | Ever told you had coronary heart disease? (1=Yes, 2=No) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| MCQ160B | Told had congestive heart failure | ⚠️ LEAK — another heart diagnosis |
| MCQ160C | Told had coronary heart disease | ⚠️ LEAK — related heart diagnosis |
| MCQ160D | Told had angina | ⚠️ LEAK — related heart diagnosis |
| MCQ010 | Told had asthma | Different disease, not our target |
| MCQ160A | Told had arthritis | Not our target |
| MCQ160M | Told had thyroid problem | Not our target |
| MCQ160L | Told had liver condition | Not our target |
| MCQ160P | Told had COPD | Not our target |
| MCQ220 | Told had cancer | Not our target |
| MCQ230A-D | Type of cancer | Not our target |
| MCQ500-560 | Liver condition details | Not our target |
| MCQ035-149, MCQ195, AGQ030 | Various other questions | Not our targets |
| OSQ230 | Osteoporosis question | Not our target |

### ⚠️ Important — this is a LABEL MAKER

- `MCQ160E` OR `MCQ160F` = Yes → person has CVD history (label = 1)
- After building the label, we **DROP these** from features
- Other heart diagnoses (B, C, D) are LEAK — drop them too

### Special codes to clean later
- Value **9** (Don't know) → remove those rows

**Rule:** Use MCQ160E and MCQ160F to build the CVD label, then drop them. Ignore all the other disease columns.

In [344]:
# Look at the smoking table
smq = data["SMQ_L"]

print("Shape:", smq.shape)
print("\nFirst 5 rows:")
smq.head()

Shape: (9015, 9)

First 5 rows:


,SEQN,SMQ020,SMQ040,SMD641,SMD650,SMD100MN,SMQ621,SMD630,SMAQUEX2
0,130378,1,3,0,0,0,0,0,1
1,130379,1,3,0,0,0,0,0,1
2,130380,2,0,0,0,0,0,0,1
3,130384,2,0,0,0,0,0,0,1
4,130385,2,0,0,0,0,0,0,1


In [345]:
# Keep smoking columns (safe lifestyle feature)
smq_clean = data["SMQ_L"][[
    "SEQN",      # unique ID
    "SMQ020",    # smoked 100+ cigarettes ever? (1=Yes, 2=No)
    "SMQ040"     # smoke now? (1=Every day, 2=Some days, 3=Not at all)
]]

print("Shape:", smq_clean.shape)
smq_clean.head()

Shape: (9015, 3)


,SEQN,SMQ020,SMQ040
0,130378,1,3
1,130379,1,3
2,130380,2,0
3,130384,2,0
4,130385,2,0


## SMQ_L — Smoking Table (Safe Lifestyle Feature)

This table has **9 columns**. We keep **2** that tell us a person's smoking status.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| SMQ020 | Smoked 100+ cigarettes in life? (1=Yes, 2=No) |
| SMQ040 | Do you smoke now? (1=Every day, 2=Some days, 3=Not at all) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| SMD641 | Days smoked in past 30 days | Too detailed — we keep simple status |
| SMD650 | Cigarettes per day | Detail — not needed |
| SMD100MN | Menthol detail | Not needed |
| SMQ621 | Cigarettes smoked in life (detail) | Detail — not needed |
| SMD630 | Age started smoking | Not needed |
| SMAQUEX2 | Questionnaire mode flag | Admin flag |

### ✅ Safe feature

Smoking is a **safe lifestyle feature** — use it for ALL 5 models. It is a genuine risk factor, not a diagnosis.

### How we'll use it later
We combine SMQ020 + SMQ040 into one simple "smoking status":
- Never smoked (SMQ020 = 2)
- Former smoker (SMQ020 = 1, SMQ040 = 3)
- Current smoker (SMQ020 = 1, SMQ040 = 1 or 2)

### Special codes to clean later
- SMQ020 = **7** (Refused) or **9** (Don't know) → remove those rows

**Rule:** Keep the 2 smoking columns. Combine them into one status later. Ignore the detailed smoking counts.

In [346]:
# Look at the alcohol table
alq = data["ALQ_L"]

print("Shape:", alq.shape)
print("\nFirst 5 rows:")
alq.head()

Shape: (6337, 9)

First 5 rows:


,SEQN,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170
0,130378,0,0,0,0,0,0,0,0
1,130379,1,2,3,0,0,0,2,0
2,130380,1,10,1,0,0,0,2,0
3,130386,1,4,2,10,0,10,2,0
4,130387,1,0,0,0,0,0,2,0


In [347]:
# Keep alcohol columns (safe lifestyle feature)
alq_clean = data["ALQ_L"][[
    "SEQN",      # unique ID
    "ALQ111",    # ever drink alcohol? (1=Yes, 2=No)
    "ALQ121",    # how often do you drink
    "ALQ130"     # average drinks per day
]]

print("Shape:", alq_clean.shape)
alq_clean.head()

Shape: (6337, 4)


,SEQN,ALQ111,ALQ121,ALQ130
0,130378,0,0,0
1,130379,1,2,3
2,130380,1,10,1
3,130386,1,4,2
4,130387,1,0,0


## ALQ_L — Alcohol Table (Safe Lifestyle Feature)

This table has **9 columns**. We keep **3** that describe drinking habits.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| ALQ111 | Ever drink alcohol? (1=Yes, 2=No) |
| ALQ121 | How often do you drink (frequency) |
| ALQ130 | Average drinks per day |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| ALQ142 | How often had 4-5+ drinks | Too detailed |
| ALQ270 | Times had 4-5+ drinks (past year) | Too detailed |
| ALQ280 | Times had 8+ drinks | Too detailed |
| ALQ151 | Ever drank 4-5 daily | Optional binge detail |
| ALQ170 | Times binged in 30 days | Too detailed |

### ✅ Safe feature

Alcohol is a **safe lifestyle feature** — use it for ALL 5 models.

### ⚠️ Special code trap (very important!)
For `ALQ130` (drinks per day):
- Values like **7 or 9 are REAL** — they mean 7 or 9 drinks (keep them!)
- Only **777** (Refused) and **999** (Don't know) are codes to remove

This is the trap from your old project — never blindly remove 7 and 9 from quantity columns!

**Rule:** Keep the 3 alcohol columns. For ALQ130, only remove 777/999, NOT 7/9.

In [348]:
# Look at the physical activity table
paq = data["PAQ_L"]

print("Shape:", paq.shape)
print("\nFirst 5 rows:")
paq.head()

Shape: (8153, 8)

First 5 rows:


,SEQN,PAD790Q,PAD790U,PAD800,PAD810Q,PAD810U,PAD820,PAD680
0,130378,3,W,45,3,W,45,360
1,130379,4,W,45,3,W,45,480
2,130380,1,W,20,0,NaN,0,240
3,130384,0,NaN,0,0,NaN,0,60
4,130385,1,D,90,1,W,60,180


In [349]:
# Keep physical activity columns (safe lifestyle feature)
paq_clean = data["PAQ_L"][[
    "SEQN",      # unique ID
    "PAD800",    # moderate activity minutes per day
    "PAD820",    # vigorous activity minutes per day
    "PAD680"     # sitting (sedentary) minutes per day
]]

print("Shape:", paq_clean.shape)
paq_clean.head()

Shape: (8153, 4)


,SEQN,PAD800,PAD820,PAD680
0,130378,45,45,360
1,130379,45,45,480
2,130380,20,0,240
3,130384,0,0,60
4,130385,90,60,180


## PAQ_L — Physical Activity Table (Safe Lifestyle Feature)

This table has **8 columns**. We keep **3** that measure exercise and sitting time.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| PAD800 | Moderate activity minutes per day |
| PAD820 | Vigorous activity minutes per day |
| PAD680 | Sitting (sedentary) minutes per day |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| PAD790Q | Activity frequency (number) | Comes in confusing number+unit pairs |
| PAD790U | Activity frequency unit (D/W/Y) | Hard to use; minutes columns are enough |
| PAD810Q | Vigorous frequency (number) | Same problem |
| PAD810U | Vigorous frequency unit | Hard to use |

### ✅ Safe feature

Physical activity is a **safe lifestyle feature** — use it for ALL 5 models. Exercise strongly affects every health condition.

### ⚠️ Special code trap
For activity minutes:
- **7 minutes is a REAL value** (keep it!)
- Only **9999** (Don't know) and **7777** (Refused) are codes to remove

### How we'll use it later
We add moderate + vigorous = total activity minutes, then group into activity levels (Sedentary / Low / Moderate / High).

**Rule:** Keep the 3 activity columns. Remove only 9999/7777, NOT 7. Ignore the confusing frequency+unit pairs.

In [350]:
# Look at the sleep table
slq = data["SLQ_L"]

print("Shape:", slq.shape)
print("\nFirst 5 rows:")
slq.head()

Shape: (8501, 7)

First 5 rows:


,SEQN,SLQ300,SLQ310,SLD012,SLQ320,SLQ330,SLD013
0,130378,21:30,07:00,9.5,00:00,09:00,9.0
1,130379,21:00,06:00,9.0,21:00,06:00,9.0
2,130380,00:00,08:00,8.0,00:00,09:00,9.0
3,130384,21:30,05:00,7.5,23:00,07:00,8.0
4,130385,22:05,06:15,8.0,22:05,06:15,8.0


In [351]:
# Keep sleep hours columns (safe lifestyle feature)
slq_clean = data["SLQ_L"][[
    "SEQN",      # unique ID
    "SLD012",    # sleep hours on weekdays
    "SLD013"     # sleep hours on weekends
]]

print("Shape:", slq_clean.shape)
slq_clean.head()

Shape: (8501, 3)


,SEQN,SLD012,SLD013
0,130378,9.5,9.0
1,130379,9.0,9.0
2,130380,8.0,9.0
3,130384,7.5,8.0
4,130385,8.0,8.0


## SLQ_L — Sleep Table (Safe Lifestyle Feature)

This table has **7 columns**. We keep **2** clean number columns for sleep hours.

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| SLD012 | Sleep hours on weekdays (clean number, 0-14) |
| SLD013 | Sleep hours on weekends (clean number, 0-14) |

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| SLQ300 | Weekday bedtime (clock time) | Text time — SLD012 hours are cleaner |
| SLQ310 | Weekday wake time (clock time) | Text time — not needed |
| SLQ320 | Weekend bedtime (clock time) | Text time — not needed |
| SLQ330 | Weekend wake time (clock time) | Text time — not needed |

### ✅ Safe feature

Sleep is a **safe lifestyle feature** — use it for ALL 5 models. Too little or too much sleep affects blood sugar, blood pressure, and weight.

### How we'll use it later
We combine weekday and weekend with a weighted average:
`(weekday × 5 + weekend × 2) / 7 = average sleep hours`
(because there are 5 weekdays and 2 weekend days)

### Good news
These columns are already clean (0-14 hours), no special codes to remove!

**Rule:** Keep the 2 sleep-hour columns. Ignore the clock-time columns. Combine into average sleep later.

In [352]:
# Look at the depression (PHQ-9) table
dpq = data["DPQ_L"]

print("Shape:", dpq.shape)
print("\nFirst 5 rows:")
dpq.head()

Shape: (6337, 11)

First 5 rows:


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,DPQ100
0,130378,0,0,0,0,0,0,0,0,0,0
1,130379,0,0,1,0,0,0,0,0,0,0
2,130380,0,0,1,1,0,0,0,0,0,0
3,130386,0,0,0,0,0,1,0,0,0,0
4,130387,0,0,0,0,0,0,0,0,0,0


In [353]:
# Keep the 9 PHQ-9 depression questions (safe lifestyle feature)
dpq_clean = data["DPQ_L"][[
    "SEQN",      # unique ID
    "DPQ010", "DPQ020", "DPQ030",   # questions 1-3
    "DPQ040", "DPQ050", "DPQ060",   # questions 4-6
    "DPQ070", "DPQ080", "DPQ090"    # questions 7-9
]]

print("Shape:", dpq_clean.shape)
dpq_clean.head()

Shape: (6337, 10)


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090
0,130378,0,0,0,0,0,0,0,0,0
1,130379,0,0,1,0,0,0,0,0,0
2,130380,0,0,1,1,0,0,0,0,0
3,130386,0,0,0,0,0,1,0,0,0
4,130387,0,0,0,0,0,0,0,0,0


## DPQ_L — Depression Table (PHQ-9) (Safe Lifestyle Feature)

This table has **11 columns**. It's the standard PHQ-9 depression screening — 9 questions, each scored 0-3. We add them into one depression score (0-27).

### Columns we KEEP

| Column | Meaning |
|---|---|
| SEQN | Unique person ID (to join tables) |
| DPQ010 | Little interest in doing things |
| DPQ020 | Feeling down or depressed |
| DPQ030 | Trouble sleeping |
| DPQ040 | Feeling tired / low energy |
| DPQ050 | Poor appetite or overeating |
| DPQ060 | Feeling bad about yourself |
| DPQ070 | Trouble concentrating |
| DPQ080 | Moving/speaking slowly or restless |
| DPQ090 | Thoughts of self-harm |

(Each scored: 0=Not at all, 1=Several days, 2=More than half the days, 3=Nearly every day)

### Columns we IGNORE

| Column | Meaning | Why skip |
|---|---|---|
| DPQ100 | How difficult these problems made life | Follow-up, not part of the 0-27 score |

### ✅ Safe feature

Depression is a **safe lifestyle feature** — use it for ALL 5 models. Mental health links to diabetes, heart disease, and weight.

### How we'll use it later
Add all 9 questions → depression_score (0-27):
- 0-4 = None/minimal
- 5-9 = Mild
- 10-14 = Moderate
- 15-27 = Moderately severe to severe

### Special codes to clean later
- Value **7** (Refused) or **9** (Don't know) → remove those rows

**Rule:** Keep the 9 PHQ-9 questions, sum them into one depression score. Ignore the follow-up question (DPQ100).

---
# ✅ STEP 1 COMPLETE — Data Loading & Column Selection

I have loaded all **20 NHANES tables** and selected only the useful columns from each.

### What I did:
- Loaded all 20 Excel files into Python
- Looked inside each table
- Kept only meaningful columns, ignored survey weights, duplicates, and flags
- Documented every KEEP and IGNORE decision

### My 20 clean tables:

| Group | Tables |
|---|---|
| Demographics | demo_clean |
| Examination | bmx_clean, bpxo_clean |
| Laboratory | ghb_clean, glu_clean, ins_clean, hdl_clean, tchol_clean, trigly_clean, hscrp_clean, biopro_clean, fastqx_clean |
| Diagnosis (labels) | diq_clean, bpq_clean, mcq_clean |
| Lifestyle | smq_clean, alq_clean, paq_clean, slq_clean, dpq_clean |

### Key principle followed:
**Label-maker columns** (HbA1c, glucose, insulin, diagnosis answers) are kept only to BUILD targets — they will be DROPPED before training to avoid data leakage.
---

---
# 🔗 STEP 2 — Joining All 20 Tables

Now I will combine all 20 tables into **ONE big table**.

### How:
- Every table has a common column: **SEQN** (unique person ID)
- I join all tables on SEQN using a **LEFT JOIN**
- I start from DEMO (everyone is in it) and attach the others one by one

### Why LEFT JOIN:
- DEMO has every participant
- Some people did not do every blood test (e.g. fasting glucose)
- LEFT JOIN keeps all people and fills missing tests with blank (NaN) — which we clean later
---

In [354]:
# ── JOIN ALL 20 TABLES ON SEQN ──

# Start with demographics (everyone is in it)
df = demo_clean.copy()

# List all the other tables to attach
other_tables = [
    bmx_clean, bpxo_clean,                          # examination
    ghb_clean, glu_clean, ins_clean, hdl_clean,     # labs
    tchol_clean, trigly_clean, hscrp_clean,
    biopro_clean, fastqx_clean,
    diq_clean, bpq_clean, mcq_clean,                # diagnosis
    smq_clean, alq_clean, paq_clean,                # lifestyle
    slq_clean, dpq_clean
]

# Attach each table one by one using LEFT JOIN on SEQN
for table in other_tables:
    df = df.merge(table, on="SEQN", how="left")

print("Final joined table shape:", df.shape)
print(f"\n{df.shape[0]} people, {df.shape[1]} columns")
df.head()

Final joined table shape: (11933, 57)

11933 people, 57 columns


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,INDFMPIR,DMDEDUC2,RIDEXPRG,BMXWT,BMXHT,BMXWAIST,...,SLD013,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090
0,130378,43,1,6,5.00,5,0,86.9,179.5,98.3,...,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,130379,66,1,3,5.00,5,0,101.8,174.2,114.7,...,9.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,130380,44,2,2,1.41,3,2,69.4,152.9,93.5,...,9.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,130381,5,2,7,1.53,0,0,34.3,120.1,70.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382,2,1,3,3.60,0,0,13.6,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
# 🔗 STEP 2 COMPLETE — All 20 Tables Joined

All 21 tables combined into ONE table using SEQN (LEFT JOIN).

**Result:** 11,933 people × 57 columns

- LEFT JOIN kept everyone from demographics
- Missing tests show as NaN (blank) — cleaned in the next step
- Children and incomplete records will be filtered next
---

## 🧹 Cleaning Filter 1 — Keep Only Adults (18+)

NHANES includes children, but we only predict **adult** diseases (diabetes, heart, hypertension, etc.).

**Why remove under-18:**
- Children rarely get these adult conditions
- Their lab values and BMI are measured differently
- Including them adds noise and confuses the model

**Action:** Keep only people aged 18 or older.

In [355]:
# Keep only adults aged 18 and above
print("Before:", df.shape[0], "people")

df = df[df["RIDAGEYR"] >= 18]

print("After (adults 18+):", df.shape[0], "people")

Before: 11933 people
After (adults 18+): 8153 people


## 🧹 Cleaning Filter 2 — Remove Pregnant Women

Pregnancy temporarily changes many health values — blood sugar, blood pressure, weight, and cholesterol all shift during pregnancy.

**Why remove pregnant women:**
- Their readings are temporary, not their normal health state
- Pregnancy can cause temporary gestational diabetes / high BP
- Including them would confuse the model about real risk factors

**Action:** Remove anyone currently pregnant (RIDEXPRG = 1).


In [356]:
# Remove pregnant women
# RIDEXPRG: 1 = pregnant, 2 = not pregnant, 3 = could not determine
print("Before:", df.shape[0], "people")

# Keep everyone EXCEPT those marked pregnant (1)
# (we keep NaN too, because most adults won't have this filled)
df = df[df["RIDEXPRG"] != 1]

print("After removing pregnant:", df.shape[0], "people")

Before: 8153 people
After removing pregnant: 8112 people


## 🧹 Cleaning Filter 3 — Drop the Pregnancy Column

We used RIDEXPRG only to filter out pregnant women. It is NOT a feature for prediction, so we remove it now.

**Action:** Drop RIDEXPRG column.

In [357]:
# Drop pregnancy column - we only needed it for filtering
df = df.drop(columns=["RIDEXPRG"])

print("Shape after dropping pregnancy column:", df.shape)
print("\nRIDEXPRG still there?", "RIDEXPRG" in df.columns)

Shape after dropping pregnancy column: (8112, 56)

RIDEXPRG still there? False


## 🧹 Cleaning Step 4 — Handle Special Codes (Refused / Don't Know)

NHANES uses special codes for "Refused" and "Don't know" answers:
- **7, 77, 777, 7777** = Refused
- **9, 99, 999, 9999** = Don't know

These are NOT real values — they must become blank (NaN).

### ⚠️ The Trap
The code size depends on the column's scale. We must remove the RIGHT code for each column — and NOT remove real values (like 7 drinks, 7 minutes, or 7 hours sleep).

### Columns we clean (with their correct codes):

| Column | Type | Codes to remove |
|---|---|---|
| DIQ010, DIQ160 (diabetes) | yes/no | 7, 9 |
| BPQ020 (hypertension) | yes/no | 7, 9 |
| MCQ160E, MCQ160F (heart) | yes/no | 7, 9 |
| SMQ020, SMQ040 (smoking) | yes/no | 7, 9 |
| ALQ111 (drink yes/no) | yes/no | 7, 9 |
| ALQ121 (drink frequency) | 0-10 scale | 77, 99 |
| ALQ130 (drinks per day) | count | 777, 999 |
| PAD800, PAD820, PAD680 (activity) | minutes | 7777, 9999 |
| DPQ010–090 (depression) | 0-3 scale | 7, 9 |

### ✅ Columns we DO NOT touch:
- **SLD012, SLD013 (sleep)** — here 7 and 9 are REAL sleep hours, not codes!

**Action:** Replace each column's specific codes with NaN. Leave sleep untouched.

In [358]:
import numpy as np

print("Cleaning special codes...\n")

# ── YES/NO and 0-3 scale columns → remove 7 and 9 ──
yesno_cols = [
    "DIQ010", "DIQ160",           # diabetes
    "BPQ020",                      # hypertension
    "MCQ160E", "MCQ160F",          # heart
    "SMQ020", "SMQ040",            # smoking
    "ALQ111",                      # drink yes/no
    "DPQ010", "DPQ020", "DPQ030",  # depression
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]
for col in yesno_cols:
    df[col] = df[col].replace([7, 9], np.nan)

# ── FREQUENCY column (0-10 scale) → remove 77 and 99 ──
df["ALQ121"] = df["ALQ121"].replace([77, 99], np.nan)

# ── DRINKS COUNT → remove 777 and 999 ──
df["ALQ130"] = df["ALQ130"].replace([777, 999], np.nan)

# ── ACTIVITY MINUTES → remove 7777 and 9999 ──
activity_cols = ["PAD800", "PAD820", "PAD680"]
for col in activity_cols:
    df[col] = df[col].replace([7777, 9999], np.nan)

# ── SLEEP (SLD012, SLD013) → NOT touched (7,9 are real hours) ──

print("Done! Special codes replaced with NaN")
print("Shape:", df.shape)

Cleaning special codes...

Done! Special codes replaced with NaN
Shape: (8112, 56)


In [359]:
# Check how much data is missing in each column
print("Missing values per column:\n")

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False)

# Show columns that have missing data
for col in missing_pct.index:
    if missing_pct[col] > 0:
        print(f"{col:12} -> {missing[col]:5} missing ({missing_pct[col]}%)")

Missing values per column:

LBXIN        ->  4578 missing (56.4%)
LBXGLU       ->  4578 missing (56.4%)
LBDLDL       ->  4578 missing (56.4%)
LBXTLG       ->  4578 missing (56.4%)
DPQ010       ->  1837 missing (22.6%)
DPQ040       ->  1824 missing (22.5%)
DPQ030       ->  1823 missing (22.5%)
DPQ020       ->  1826 missing (22.5%)
DPQ080       ->  1828 missing (22.5%)
ALQ130       ->  1829 missing (22.5%)
BPXOPLS1     ->  1816 missing (22.4%)
BPXOPLS2     ->  1816 missing (22.4%)
LBXGH        ->  1816 missing (22.4%)
BPXOPLS3     ->  1816 missing (22.4%)
BPXOSY1      ->  1816 missing (22.4%)
BPXOSY2      ->  1816 missing (22.4%)
BPXODI3      ->  1816 missing (22.4%)
BPXODI2      ->  1816 missing (22.4%)
BMXWT        ->  1816 missing (22.4%)
BMXHT        ->  1816 missing (22.4%)
BMXBMI       ->  1816 missing (22.4%)
BMXWAIST     ->  1816 missing (22.4%)
BPXODI1      ->  1816 missing (22.4%)
BPXOSY3      ->  1816 missing (22.4%)
DPQ070       ->  1819 missing (22.4%)
DPQ060       ->  1821 

---
# 🎯 STEP — Building the 4 Target Labels
We create the 4 answers (Y) our models will predict.
Each label is built from diagnosis + medical thresholds.
---

In [360]:
# Label 1: Diabetes
df["diabetes"] = ((df["DIQ010"] == 1) | (df["LBXGH"] >= 6.5)).astype(int)
print("Diabetes:", dict(df["diabetes"].value_counts()))

Diabetes: {0: np.int64(6887), 1: np.int64(1225)}


In [361]:
# Label 2: Heart disease (CVD)
df["cvd"] = ((df["MCQ160E"] == 1) | (df["MCQ160F"] == 1)).astype(int)
print("CVD:", dict(df["cvd"].value_counts()))

CVD: {0: np.int64(7478), 1: np.int64(634)}


In [362]:
# Label 3: Hypertension
df["hypertension"] = (df["BPQ020"] == 1).astype(int)
print("Hypertension:", dict(df["hypertension"].value_counts()))

Hypertension: {0: np.int64(5160), 1: np.int64(2952)}


In [363]:
# Label 4: Metabolic
import numpy as np

sys_avg = df[["BPXOSY1","BPXOSY2","BPXOSY3"]].mean(axis=1)
dia_avg = df[["BPXODI1","BPXODI2","BPXODI3"]].mean(axis=1)

c1 = ((df["RIAGENDR"]==1)&(df["BMXWAIST"]>=102)) | ((df["RIAGENDR"]==2)&(df["BMXWAIST"]>=88))
c2 = df["LBXTLG"] >= 150
c3 = ((df["RIAGENDR"]==1)&(df["LBDHDD"]<40)) | ((df["RIAGENDR"]==2)&(df["LBDHDD"]<50))
c4 = (sys_avg >= 130) | (dia_avg >= 85)
c5 = df["LBXGLU"] >= 100

criteria = c1.astype(int)+c2.astype(int)+c3.astype(int)+c4.astype(int)+c5.astype(int)
df["metabolic"] = (criteria >= 3).astype(int)
print("Metabolic:", dict(df["metabolic"].value_counts()))

Metabolic: {0: np.int64(6766), 1: np.int64(1346)}


---
# 🔧 Feature Engineering
Turn messy raw columns into clean features:
- Average 3 BP readings → systolic_avg, diastolic_avg, pulse_avg
- Combine sleep → avg_sleep_hours
- Sum depression → depression_score
- Total activity, smoking status, alcohol
---

In [364]:
# ── FEATURE ENGINEERING ──

# 1. Average 3 BP readings
df["systolic_avg"] = df[["BPXOSY1","BPXOSY2","BPXOSY3"]].mean(axis=1)
df["diastolic_avg"] = df[["BPXODI1","BPXODI2","BPXODI3"]].mean(axis=1)
df["pulse_avg"] = df[["BPXOPLS1","BPXOPLS2","BPXOPLS3"]].mean(axis=1)

# 2. Average sleep (weekday counts more)
df["avg_sleep_hours"] = (df["SLD012"] * 5 + df["SLD013"] * 2) / 7

# 3. Sum 9 depression questions
dpq_cols = ["DPQ010","DPQ020","DPQ030","DPQ040","DPQ050",
            "DPQ060","DPQ070","DPQ080","DPQ090"]
df["depression_score"] = df[dpq_cols].sum(axis=1)

# 4. Total activity
df["total_activity"] = df["PAD800"].fillna(0) + df["PAD820"].fillna(0)

# 5. Smoking status: 0=never, 1=former, 2=current
def smoking_status(row):
    if row["SMQ020"] == 2:
        return 0
    elif row["SMQ040"] == 3:
        return 1
    elif row["SMQ040"] in [1, 2]:
        return 2
    else:
        return np.nan
df["smoking_status"] = df.apply(smoking_status, axis=1)

# 6. Alcohol: drinks per day (0 if never)
df["alcohol_drinks"] = df["ALQ130"]
df.loc[df["ALQ111"] == 2, "alcohol_drinks"] = 0

print("Feature engineering done!")
print("Shape:", df.shape)

Feature engineering done!
Shape: (8112, 68)


---
# ✂️ Drop Raw Columns
We replaced these raw columns with clean engineered features, so remove the raw ones:
- BP readings → systolic_avg, diastolic_avg, pulse_avg
- Sleep → avg_sleep_hours
- Depression 9 questions → depression_score
- Activity → total_activity
- Smoking → smoking_status
- Alcohol → alcohol_drinks
---

In [365]:
# ── DROP RAW COLUMNS ──
raw_columns_to_drop = [
    "BPXOSY1","BPXOSY2","BPXOSY3",
    "BPXODI1","BPXODI2","BPXODI3",
    "BPXOPLS1","BPXOPLS2","BPXOPLS3",
    "SLD012","SLD013",
    "DPQ010","DPQ020","DPQ030","DPQ040","DPQ050",
    "DPQ060","DPQ070","DPQ080","DPQ090",
    "PAD800","PAD820","PAD680",
    "SMQ020","SMQ040",
    "ALQ111","ALQ121","ALQ130"
]
df = df.drop(columns=raw_columns_to_drop)
print("Raw columns dropped!")
print("Shape now:", df.shape)

Raw columns dropped!
Shape now: (8112, 40)


In [366]:
# Drop fasting hours - testing helper, not a real feature
df = df.drop(columns=["PHAFSTHR"])
print("PHAFSTHR dropped!")
print("Shape now:", df.shape)

PHAFSTHR dropped!
Shape now: (8112, 39)


---
# 🛠️ Per-Model Setup — Leakage Prevention
Define what each model drops to stay leakage-free, plus a helper
function that builds clean X (features) and y (label) for any model.
---

In [367]:
# ── MODEL SETUP (SCREENING VERSION - cheap inputs only) ──
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import numpy as np

ALL_LABELS = ["diabetes", "cvd", "hypertension", "metabolic"]

# Expensive lab tests - these are the "go get tested" outputs, NOT inputs
EXPENSIVE_LABS = [
    "LBXGH", "LBXGLU", "LBXIN",          # diabetes tests
    "LBDHDD", "LBXTC", "LBXTLG", "LBDLDL", # cholesterol panel
    "LBXHSCRP",                            # inflammation
    "LBXSCR", "LBXSUA", "LBXSBU", "LBXSAL" # kidney/liver
]

# Always dropped: ID, labels, diagnosis, AND all expensive labs
ALWAYS_DROP = ["SEQN"] + ALL_LABELS + [
    "DIQ010", "DIQ160", "BPQ020", "MCQ160E", "MCQ160F"
] + EXPENSIVE_LABS

# Leakage columns per model (body-size leakage for metabolic)
LEAKAGE = {
    "diabetes":     [],
    "cvd":          [],
    "hypertension": ["systolic_avg", "diastolic_avg"],
    "metabolic":    ["BMXWAIST", "systolic_avg", "diastolic_avg"],
}

print("Screening setup done!")
print("Expensive labs excluded from inputs:", len(EXPENSIVE_LABS))

Screening setup done!
Expensive labs excluded from inputs: 12


In [368]:
# ── HELPER: build clean X and y for any model ──
def build_model_data(model_name):
    data = df.copy()

    # Metabolic: keep only fasting people
    if model_name == "metabolic":
        data = data[data["LBXGLU"].notna() & data["LBXTLG"].notna()]

    y = data[model_name]

    # Remove rows where label is missing
    valid = y.notna()
    data = data[valid]
    y = y[valid]

    # Build features: drop leakage + always-drop
    drop_cols = ALWAYS_DROP + LEAKAGE[model_name]
    X = data.drop(columns=[c for c in drop_cols if c in data.columns])

    # Fill small gaps with median
    X = X.fillna(X.median())

    return X, y

# Verify all 5 models are leakage-free
for model in ALL_LABELS:
    X, y = build_model_data(model)
    leak_check = [c for c in LEAKAGE[model] if c in X.columns]
    status = "✅ clean" if not leak_check else f"⚠️ LEAK: {leak_check}"
    print(f"{model:13}: {X.shape[0]} people, {X.shape[1]} features, {status}")

diabetes     : 8112 people, 17 features, ✅ clean
cvd          : 8112 people, 17 features, ✅ clean
hypertension : 8112 people, 15 features, ✅ clean
metabolic    : 3534 people, 14 features, ✅ clean


---
# 🤖 Training All 4 Models
For each model we train 3 algorithms (Logistic Regression, Random Forest, XGBoost)
with regularization and cross-validation, then pick the best by Test AUC.

### Good-fitting checks:
- Train/Test split (80/20)
- 5-fold cross-validation
- Regularized models (limited depth)
- Train vs Test gap (small gap = good fit, no overfitting)
---

In [369]:
# ════════════════════════════════════════════════════
#  TRAIN DIABETES MODEL
# ════════════════════════════════════════════════════
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get clean diabetes data (leakage already removed)
X, y = build_model_data("diabetes")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 3 algorithms — DIABETES settings
algos = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_leaf=20,
        class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, eval_metric="logloss"),
}

print(f"\n{'='*55}")
print(f"  DIABETES  ({X_train.shape[0]} train, {X_test.shape[0]} test)")
print(f"{'='*55}")

best_auc, best_name = 0, None
for name, model in algos.items():
    if name == "LogReg":
        model.fit(X_train_s, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train_s)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test_s)[:,1])
        cvs = cross_val_score(model, X_train_s, y_train, cv=cv, scoring="roc_auc")
    else:
        model.fit(X_train, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        cvs = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

    gap = tr - te
    fit = "✅" if gap < 0.05 else "⚠️"
    print(f"  {name:13} Train={tr:.3f}  Test={te:.3f}  CV={cvs.mean():.3f}  Gap={gap:.3f} {fit}")

    if te > best_auc:
        best_auc, best_name = te, name

print(f"  → Best: {best_name} (Test AUC = {best_auc:.3f})")


  DIABETES  (6489 train, 1623 test)


  LogReg        Train=0.794  Test=0.800  CV=0.790  Gap=-0.005 ✅
  RandomForest  Train=0.858  Test=0.806  CV=0.800  Gap=0.052 ⚠️
  XGBoost       Train=0.862  Test=0.819  CV=0.818  Gap=0.042 ✅
  → Best: XGBoost (Test AUC = 0.819)


In [370]:
# ════════════════════════════════════════════════════
#  TRAIN CVD (HEART DISEASE) MODEL
#  Tighter settings - rare disease (only 8% positive)
# ════════════════════════════════════════════════════
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get clean CVD data (leakage already removed)
X, y = build_model_data("cvd")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 3 algorithms — CVD settings (TIGHTER to reduce overfitting)
algos = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=40,   # tighter
        class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=2, learning_rate=0.02,    # tighter
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=10, reg_alpha=1.0, reg_lambda=3.0,   # more penalty
        random_state=42, eval_metric="logloss"),
}

print(f"\n{'='*55}")
print(f"  CVD (HEART DISEASE)  ({X_train.shape[0]} train, {X_test.shape[0]} test)")
print(f"{'='*55}")

best_auc, best_name = 0, None
for name, model in algos.items():
    if name == "LogReg":
        model.fit(X_train_s, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train_s)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test_s)[:,1])
        cvs = cross_val_score(model, X_train_s, y_train, cv=cv, scoring="roc_auc")
    else:
        model.fit(X_train, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        cvs = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

    gap = tr - te
    fit = "✅" if gap < 0.05 else "⚠️"
    print(f"  {name:13} Train={tr:.3f}  Test={te:.3f}  CV={cvs.mean():.3f}  Gap={gap:.3f} {fit}")

    if te > best_auc:
        best_auc, best_name = te, name

print(f"  → Best: {best_name} (Test AUC = {best_auc:.3f})")


  CVD (HEART DISEASE)  (6489 train, 1623 test)
  LogReg        Train=0.815  Test=0.795  CV=0.807  Gap=0.021 ✅
  RandomForest  Train=0.849  Test=0.802  CV=0.810  Gap=0.047 ✅
  XGBoost       Train=0.838  Test=0.810  CV=0.815  Gap=0.028 ✅
  → Best: XGBoost (Test AUC = 0.810)


In [371]:
# ════════════════════════════════════════════════════
#  TRAIN HYPERTENSION MODEL
#  Standard settings - well balanced (36% positive)
# ════════════════════════════════════════════════════
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get clean hypertension data (leakage already removed - BP readings dropped)
X, y = build_model_data("hypertension")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 3 algorithms — HYPERTENSION settings (standard)
algos = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_leaf=20,
        class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, eval_metric="logloss"),
}

print(f"\n{'='*55}")
print(f"  HYPERTENSION  ({X_train.shape[0]} train, {X_test.shape[0]} test)")
print(f"{'='*55}")

best_auc, best_name = 0, None
for name, model in algos.items():
    if name == "LogReg":
        model.fit(X_train_s, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train_s)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test_s)[:,1])
        cvs = cross_val_score(model, X_train_s, y_train, cv=cv, scoring="roc_auc")
    else:
        model.fit(X_train, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        cvs = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

    gap = tr - te
    fit = "✅" if gap < 0.05 else "⚠️"
    print(f"  {name:13} Train={tr:.3f}  Test={te:.3f}  CV={cvs.mean():.3f}  Gap={gap:.3f} {fit}")

    if te > best_auc:
        best_auc, best_name = te, name

print(f"  → Best: {best_name} (Test AUC = {best_auc:.3f})")


  HYPERTENSION  (6489 train, 1623 test)
  LogReg        Train=0.782  Test=0.787  CV=0.780  Gap=-0.005 ✅
  RandomForest  Train=0.815  Test=0.792  CV=0.781  Gap=0.023 ✅
  XGBoost       Train=0.818  Test=0.796  CV=0.790  Gap=0.022 ✅
  → Best: XGBoost (Test AUC = 0.796)


In [372]:
# ════════════════════════════════════════════════════
#  TRAIN METABOLIC SYNDROME MODEL
#  Uses only fasting people (3,534) - criteria columns dropped
# ════════════════════════════════════════════════════
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get clean metabolic data (5 criteria columns already dropped)
X, y = build_model_data("metabolic")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 3 algorithms — METABOLIC settings (slightly tighter, smaller data)
algos = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=5, min_samples_leaf=25,
        class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, eval_metric="logloss"),
}

print(f"\n{'='*55}")
print(f"  METABOLIC SYNDROME  ({X_train.shape[0]} train, {X_test.shape[0]} test)")
print(f"{'='*55}")

best_auc, best_name = 0, None
for name, model in algos.items():
    if name == "LogReg":
        model.fit(X_train_s, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train_s)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test_s)[:,1])
        cvs = cross_val_score(model, X_train_s, y_train, cv=cv, scoring="roc_auc")
    else:
        model.fit(X_train, y_train)
        tr = roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
        te = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
        cvs = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

    gap = tr - te
    fit = "✅" if gap < 0.05 else "⚠️"
    print(f"  {name:13} Train={tr:.3f}  Test={te:.3f}  CV={cvs.mean():.3f}  Gap={gap:.3f} {fit}")

    if te > best_auc:
        best_auc, best_name = te, name

print(f"  → Best: {best_name} (Test AUC = {best_auc:.3f})")


  METABOLIC SYNDROME  (2827 train, 707 test)
  LogReg        Train=0.771  Test=0.796  CV=0.766  Gap=-0.025 ✅
  RandomForest  Train=0.825  Test=0.802  CV=0.777  Gap=0.023 ✅
  XGBoost       Train=0.836  Test=0.801  CV=0.776  Gap=0.035 ✅
  → Best: RandomForest (Test AUC = 0.802)


---
# ✅ FINAL RESULTS — Screening Models (Cheap Inputs Only)

| Disease | Best Model | Test AUC | Gap |
|---|---|---|---|
| Diabetes | XGBoost | 0.819 | 0.042 ✅ |
| CVD (Heart) | XGBoost | 0.810 | 0.028 ✅ |
| Hypertension | XGBoost | 0.796 | 0.022 ✅ |
| Metabolic Syndrome | RandomForest | 0.802 | 0.023 ✅ |

### This is a SCREENING tool:
- Uses only CHEAP inputs: age, body measurements, BP, lifestyle
- NO expensive blood tests required as inputs
- Flags high-risk patients → doctor orders confirmatory lab tests

### Honest & leakage-free:
- All gaps under 0.05 (no overfitting)
- No lab values, no diagnosis, no label-makers used as features
- Metabolic trained on fasting participants only

---
# 💾 Save the Best Models
We save each model's best algorithm + its feature list so we can
use them later in an app without retraining.

Best models:
- Diabetes → XGBoost
- CVD → XGBoost
- Hypertension → XGBoost
- Metabolic → RandomForest

In [374]:
# ── SAVE THE 4 BEST MODELS ──
import joblib
import os

# Create a folder for models
os.makedirs("models", exist_ok=True)

# Best algorithm for each model (from our results)
BEST = {
    "diabetes":     "XGBoost",
    "cvd":          "XGBoost",
    "hypertension": "XGBoost",
    "metabolic":    "RandomForest",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def get_model(name, model_name):
    """Return a fresh model with the right settings."""
    if name == "LogReg":
        return LogisticRegression(max_iter=1000, class_weight="balanced")
    elif name == "RandomForest":
        # metabolic uses tighter settings
        if model_name == "metabolic":
            return RandomForestClassifier(n_estimators=100, max_depth=5,
                   min_samples_leaf=25, class_weight="balanced", random_state=42)
        return RandomForestClassifier(n_estimators=100, max_depth=6,
               min_samples_leaf=20, class_weight="balanced", random_state=42)
    else:  # XGBoost
        if model_name == "cvd":  # tighter for rare disease
            return XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.02,
                   subsample=0.7, colsample_bytree=0.7, min_child_weight=10,
                   reg_alpha=1.0, reg_lambda=3.0, random_state=42, eval_metric="logloss")
        return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.03,
               subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
               reg_alpha=0.5, reg_lambda=2.0, random_state=42, eval_metric="logloss")

# Train on ALL data and save each best model
for model_name in ALL_LABELS:
    X, y = build_model_data(model_name)
    best_algo = BEST[model_name]
    model = get_model(best_algo, model_name)

    # LogReg needs scaling
    if best_algo == "LogReg":
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        model.fit(X_scaled, y)
        joblib.dump(scaler, f"models/{model_name}_scaler.pkl")
    else:
        model.fit(X, y)

    # Save model + feature list
    joblib.dump(model, f"models/{model_name}_model.pkl")
    joblib.dump(list(X.columns), f"models/{model_name}_features.pkl")

    print(f"✅ Saved {model_name} ({best_algo}) - {X.shape[1]} features")

print("\nAll 4 models saved in the 'models' folder!")

✅ Saved diabetes (XGBoost) - 17 features
✅ Saved cvd (XGBoost) - 17 features
✅ Saved hypertension (XGBoost) - 15 features
✅ Saved metabolic (RandomForest) - 14 features

All 4 models saved in the 'models' folder!


---
# 📊 Feature Importance — What Drives Each Prediction
For each disease, we see which inputs matter most.
This confirms the models learned medically sensible patterns
(e.g. age and BMI should matter for diabetes).
---

In [376]:
# ── FEATURE IMPORTANCE FOR EACH MODEL ──
import joblib
import pandas as pd

for model_name in ALL_LABELS:
    model = joblib.load(f"models/{model_name}_model.pkl")
    features = joblib.load(f"models/{model_name}_features.pkl")

    # Tree models have feature_importances_
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
        imp_df = pd.DataFrame({
            "feature": features,
            "importance": importance
        }).sort_values("importance", ascending=False)

        print(f"\n{'='*45}")
        print(f"  {model_name.upper()} — Top 8 drivers")
        print(f"{'='*45}")
        for _, row in imp_df.head(8).iterrows():
            bar = "█" * int(row["importance"] * 50)
            print(f"  {row['feature']:18} {bar} {row['importance']:.3f}")


  DIABETES — Top 8 drivers
  RIDAGEYR           █████████ 0.190
  total_activity     █████ 0.106
  BMXWAIST           ████ 0.088
  BMXBMI             ███ 0.073
  DMDEDUC2           ███ 0.073
  alcohol_drinks     ███ 0.062
  pulse_avg          ██ 0.054
  RIDRETH3           ██ 0.050

  CVD — Top 8 drivers
  RIDAGEYR           ██████████ 0.220
  total_activity     █████ 0.111
  smoking_status     ████ 0.099
  DMDEDUC2           ███ 0.077
  RIAGENDR           ███ 0.064
  avg_sleep_hours    ███ 0.062
  INDFMPIR           ██ 0.048
  depression_score   ██ 0.047

  HYPERTENSION — Top 8 drivers
  RIDAGEYR           ████████████████ 0.322
  BMXBMI             ████ 0.089
  total_activity     ████ 0.084
  BMXWAIST           ███ 0.066
  DMDEDUC2           ███ 0.063
  BMXWT              ██ 0.056
  RIDRETH3           ██ 0.055
  smoking_status     ██ 0.052

  METABOLIC — Top 8 drivers
  BMXBMI             ████████████████████ 0.419
  BMXWT              █████████ 0.199
  RIDAGEYR           ███████ 0.1

### Limitation — Family History Not Available
Family medical history (a known risk factor for diabetes
and heart disease) was collected in earlier NHANES cycles
(variables MCQ300A/B/C) but was DROPPED in the 2021-2023
cycle. It could not be included in these models.

If using older NHANES data (e.g. 2017-2020), family history
could be added as a valuable extra risk feature.